<a href="https://colab.research.google.com/github/Swasthika3/DL-24BAD121-/blob/main/EXNO_9_DL_24BAD121.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Expt 9: Transformer-Based LLM Chatbot using Gradio
Name: Swasthika M
Roll no: 24BAD121


In [1]:
# Setup: install libraries
!pip install -q transformers accelerate gradio

In [2]:
print("SWASTHIKA M 24BAD121")

SWASTHIKA M 24BAD121


## Part A: Loading and configuring a pre-trained LLM

In [3]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print("Model loaded:", MODEL_NAME)

Device: cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct


In [4]:
# Tokenization demo
sample = "Explain what a Transformer is in two sentences."
ids = tokenizer(sample)["input_ids"]
print("Text   :", sample)
print("Tokens :", tokenizer.convert_ids_to_tokens(ids))
print("IDs    :", ids)
print("Count  :", len(ids))

Text   : Explain what a Transformer is in two sentences.
Tokens : ['Ex', 'plain', 'Ġwhat', 'Ġa', 'ĠTransformer', 'Ġis', 'Ġin', 'Ġtwo', 'Ġsentences', '.']
IDs    : [840, 20772, 1128, 264, 62379, 374, 304, 1378, 22870, 13]
Count  : 10


In [5]:
def generate(prompt, temperature=0.7, max_new_tokens=128):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

prompts = [
    "Explain what a Transformer is in two sentences.",
    "Write a short poem about the monsoon.",
    "What is the capital of Tamil Nadu?",
]

for p in prompts:
    print("Prompt:", p)
    print("Response:", generate(p, temperature=0.7, max_new_tokens=100), "\n")

Prompt: Explain what a Transformer is in two sentences.
Response: A Transformer is an advanced neural network architecture that leverages the power of self-attention mechanisms to process sequential data, enabling it to learn long-range dependencies and high-dimensional representations efficiently. 

Prompt: Write a short poem about the monsoon.
Response: In the monsoon season, the sky is painted in hues of gold and blue,
As the sun warms the earth with its warmth.
The clouds roll in from afar,
And pour down on us in a heavy rain.

The waves crash against the shore,
And the sea swells up like an ocean tide.
We stand atop our boats,
And watch as the winds howl and rush around us.

The trees sway and sway,
As we feel the cool breeze through our hair.
We breathe in 

Prompt: What is the capital of Tamil Nadu?
Response: The capital of Tamil Nadu is Chennai.

Chennai, officially known as the "City of Lakes," is the largest city in India and serves as the administrative, commercial, industri

In [7]:
# Effect of temperature

p = "Write a one-line slogan for a coffee shop."
for t in [0.1, 0.7, 1.2]:
    print(f"\nTemperature = {t}")
    print(generate(p, temperature=t, max_new_tokens=40))

# Effect of maximum output length
p = "Describe artificial intelligence."
for m in [20, 80]:
    print(f"\nmax_new_tokens = {m}")
    print(generate(p, temperature=0.7, max_new_tokens=m))


Temperature = 0.1
" Brew your own, enjoy the experience."

Temperature = 0.7
" Brew your coffee, experience the moment."

Temperature = 1.2
"Every sip, every taste – a story of warmth and connection."

max_new_tokens = 20
Artificial intelligence is a subset of computer science that deals with the development and deployment of intelligent machines,

max_new_tokens = 80
Artificial intelligence (AI) is a branch of computer science that aims to create intelligent machines capable of performing tasks that typically require human intelligence, such as recognizing patterns in data, understanding natural language, and making decisions based on limited information. AI involves the development of algorithms, machine learning models, and neural networks that can process and analyze large amounts of data to generate insights and make predictions.
Some key characteristics


## Part B: Implementing chatbot functionality (with conversation history)

In [8]:
SYSTEM_PROMPT = "You are a helpful, concise assistant."
MAX_HISTORY_TURNS = 6   # keep the last N user/assistant pairs

def chat(user_message, history=None, temperature=0.7, max_new_tokens=256):
    """history: list of {"role": ..., "content": ...}. Returns (reply, history, latency)."""
    history = list(history or [])
    history.append({"role": "user", "content": user_message})
    trimmed = history[-2 * MAX_HISTORY_TURNS:]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + trimmed

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=temperature > 0,
            temperature=float(temperature) if temperature > 0 else None,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - start

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": reply})
    return reply, history, latency

In [9]:
# Test conversation memory and other query types

hist = []
for q in [
    "My name is Arun.",
    "What is my name?",                              # memory test
    "Who won the 2030 FIFA World Cup?",              # hallucination test
    "Write a Python function to reverse a string.",  # code
    "Summarize the plot of Romeo and Juliet in 3 lines.",
]:
    reply, hist, lat = chat(q, hist)
    print(f"You: {q}\nBot: {reply}\n[{lat:.1f}s]\n" + "-" * 60)

You: My name is Arun.
Bot: Hello! How can I assist you today?
[2.5s]
------------------------------------------------------------
You: What is my name?
Bot: Your name is Arun.
[2.2s]
------------------------------------------------------------
You: Who won the 2030 FIFA World Cup?
Bot: The 2030 FIFA World Cup was won by Argentina, who defeated Croatia in the final.
[6.3s]
------------------------------------------------------------
You: Write a Python function to reverse a string.
Bot: Here's an example of how to write a Python function to reverse a string:

```python
def reverse_string(s):
    return s[::-1]
```

This function takes a string `s` as input and returns a new string that is the reverse of `s`. The syntax `s[::-1]` means "start at the end of the string, move backwards, and stop at the first character".
[25.8s]
------------------------------------------------------------
You: Summarize the plot of Romeo and Juliet in 3 lines.
Bot: Romeo and Juliet are two young lovers who f

## Part C: Interactive interface using Gradio

In [11]:
import gradio as gr

def _as_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return str(content)

def respond(message, history, temperature, max_new_tokens):
    clean = [{"role": m["role"], "content": _as_text(m["content"])} for m in (history or [])]
    reply, _, latency = chat(message, clean, temperature, max_new_tokens)
    return f"{reply}\n\n*(response time: {latency:.1f}s)*"

demo = gr.ChatInterface(
    fn=respond,
    title="Transformer LLM Chatbot",
    description=f"Model: {MODEL_NAME}",
    additional_inputs=[
        gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(32, 512, value=256, step=16, label="Max new tokens"),
    ],
    examples=[
        ["Explain self-attention in simple words.", 0.7, 256],
        ["Write a haiku about Coimbatore.", 1.0, 64],
        ["What is the capital of France?", 0.1, 64],
    ],
)
demo.launch(share=True, debug=False)   # opens inline + gives a public link

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed17867f7b9da08ebd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Part D: Testing, organizing and deploying

Run the next cells to generate the project files (`app.py`, `part_b_chatbot.py`, `requirements.txt`, `README.md`), then upload them to GitHub and Hugging Face Spaces.

In [12]:
%%writefile requirements.txt
torch
transformers>=4.44
accelerate
gradio>=5.0,<6.0

Writing requirements.txt


In [13]:
%%writefile part_b_chatbot.py
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM_PROMPT = "You are a helpful, concise assistant."
MAX_HISTORY_TURNS = 6

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
model.eval()

def chat(user_message, history=None, temperature=0.7, max_new_tokens=256):
    history = list(history or [])
    history.append({"role": "user", "content": user_message})
    trimmed = history[-2 * MAX_HISTORY_TURNS:]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + trimmed
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=int(max_new_tokens),
            do_sample=temperature > 0,
            temperature=float(temperature) if temperature > 0 else None,
            top_p=0.9, repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - start
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": reply})
    return reply, history, latency

Writing part_b_chatbot.py


In [14]:
%%writefile app.py
import gradio as gr
from part_b_chatbot import chat, MODEL_NAME


def _as_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return str(content)

def respond(message, history, temperature, max_new_tokens):
    clean = [{"role": m["role"], "content": _as_text(m["content"])} for m in (history or [])]
    reply, _, latency = chat(message, clean, temperature, max_new_tokens)
    return f"{reply}\n\n*(response time: {latency:.1f}s)*"

demo = gr.ChatInterface(
    fn=respond,
    title="Transformer LLM Chatbot",
    description=f"Model: {MODEL_NAME} | Roll No(s): {ROLL_NOS}",
    additional_inputs=[
        gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(32, 512, value=256, step=16, label="Max new tokens"),
    ],
)

if __name__ == "__main__":
    demo.launch()

Writing app.py


In [15]:
%%writefile README.md
---
title: Transformer LLM Chatbot
emoji: 💬
sdk: gradio
sdk_version: 5.49.1
app_file: app.py
pinned: false
---

# Transformer-Based LLM Chatbot (Expt 9)
Chatbot using Hugging Face Transformers (Qwen2.5-0.5B-Instruct) with a Gradio UI.

Run locally: `pip install -r requirements.txt && python app.py`

## Limitations
- May hallucinate facts; small model with limited knowledge.
- CPU inference is slow.
- Limited context window for long conversations.

Writing README.md


In [16]:
# Download all project files as a zip (upload them to GitHub / HF Spaces)
!zip -q llm_chatbot.zip app.py part_b_chatbot.py requirements.txt README.md
from google.colab import files
files.download("llm_chatbot.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>